<a href="https://colab.research.google.com/github/tawheedul-islam/iot-botnet-brb/blob/main/IoT_BRB_Part5b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 5b: আরও ৩টা source device (২, ৬, ৯)
**লক্ষ্য:** আগের ৪টা source (১, ৪, ৫, ৮)-এর সঙ্গে আরও ৩টা যোগ করে মোট ৭টা source-এ BRB-র স্থিতিশীলতার দাবি যাচাই করা। কোডে কিছু বদলানো হয়নি, শুধু `SOURCES` ও আউটপুট ফাইলের নাম (`part5b_*.csv`)।

⏱ ~৩০ মিনিট। Runtime → Run all। শেষে `results/part5b_auc.csv` ও `results/part5b_selective.csv` নামাও।

## 1. Setup

In [1]:
import glob, os, itertools, time, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

SEED = 42
SOURCES = [2, 6, 9]                      # Part 4-এ ১,৫ আর Part 5-এ ৪,৮ হয়ে গেছে
MIRAI_DEVS = [1, 2, 4, 5, 6, 8, 9]
N_PER_CLASS = 20000
SELECTED = ['H_L0.01_weight', 'H_L3_weight', 'HpHp_L0.1_weight',
            'HH_jit_L0.01_mean', 'MI_dir_L5_variance', 'HpHp_L0.01_radius']
MISSING_RATES = [0.0, 0.3, 0.5]
N_REPEATS = 3

In [2]:
!pip install -q kagglehub
import kagglehub
path = kagglehub.dataset_download("mkashifn/nbaiot-dataset")
print(path)

Using Colab cache for faster access to the 'nbaiot-dataset' dataset.
/kaggle/input/nbaiot-dataset


## 2. Data helper

In [3]:
def find_files(dev, kind):
    return sorted(glob.glob(f"{path}/**/{dev}.{kind}*.csv", recursive=True))

def read(f):
    return pd.read_csv(f, usecols=SELECTED)[SELECTED].astype("float32")

def load_benign(dev, n=N_PER_CLASS, seed=SEED):
    d = pd.concat([read(f) for f in find_files(dev, "benign")], ignore_index=True)
    return d.sample(min(len(d), n), random_state=seed)

def load_attack(dev, family, n=N_PER_CLASS, seed=SEED):
    files = find_files(dev, family)
    assert files, f"device {dev}-এ {family} নেই"
    per = max(1, n // len(files))
    parts = []
    for f in files:
        d = read(f)
        parts.append(d.sample(min(len(d), per), random_state=seed))
    return pd.concat(parts, ignore_index=True)

def make_xy(ben, att):
    X = np.vstack([ben.values, att.values]).astype("float64")
    y = np.r_[np.zeros(len(ben)), np.ones(len(att))]
    return X, y

def build_scenario(family, dev):
    """Cross-device scenario: target device-এর benign + নতুন device-এর attack।"""
    ben = load_benign(dev, 10000).values.astype("float64")
    att = load_attack(dev, "mirai" if family == "Mirai" else "gafgyt", 10000).values.astype("float64")
    return np.vstack([ben, att]), np.r_[np.zeros(len(ben)), np.ones(len(att))]

## 3. BRB engine (Part 4-এর মতো, সঙ্গে ER যাচাই)

In [4]:
GRADES = ["Low", "Medium", "High"]

def slog(X):
    return np.sign(X) * np.log1p(np.abs(X))

class BRB:
    def __init__(self, d, lam=5.0):
        self.d, self.lam = d, lam
        self.idx = np.array(list(itertools.product(range(3), repeat=d)))
        self.K = len(self.idx)

    def match(self, Z, refs):
        n, d = Z.shape
        M = np.empty((n, d, 3))
        for i in range(d):
            r0, r1, r2 = refs[i]
            z = np.clip(Z[:, i], r0, r2)
            t1 = (z - r0) / (r1 - r0 + 1e-9)
            t2 = (z - r1) / (r2 - r1 + 1e-9)
            lo = z <= r1
            M[:, i, 0] = np.where(lo, 1 - t1, 0)
            M[:, i, 1] = np.where(lo, t1, 1 - t2)
            M[:, i, 2] = np.where(lo, 0, t2)
        return np.clip(M, 0, 1)

    def fill(self, M, Z):
        miss = np.isnan(Z)
        if not miss.any():
            return M
        return np.where(miss[:, :, None], np.ones((1, 1, 3)), M)   # missing = ignorance

    def activation(self, M, delta):
        P = M ** delta[None, :, None]
        A = P[:, 0, :]
        for i in range(1, self.d):
            A = (A[:, :, None] * P[:, i, :][:, None, :]).reshape(len(A), -1)
        return A / (A.sum(1, keepdims=True) + 1e-12)

    def fit_beliefs(self, M, y, delta, chunk=4000):
        S = np.zeros(self.K); C1 = np.zeros(self.K)
        for s in range(0, len(y), chunk):
            A = self.activation(M[s:s + chunk], delta)
            S += A.sum(0); C1 += A.T @ y[s:s + chunk]
        return np.stack([S - C1, C1], 1) / (S[:, None] + self.lam)

    def infer(self, M, delta, beta, chunk=3000):
        s = beta.sum(1); out = []
        for st in range(0, len(M), chunk):
            A = self.activation(M[st:st + chunk], delta)
            one_m = 1 - A * s[None, :]
            f = A[:, :, None] * beta[None, :, :] + one_m[:, :, None]
            P = f.prod(1); base = one_m.prod(1); wprod = (1 - A).prod(1)
            mu = 1.0 / (P.sum(1) - base + 1e-12)
            out.append(mu[:, None] * (P - base[:, None]) / (1 - mu * wprod + 1e-12)[:, None])
        b = np.clip(np.vstack(out), 0, 1)
        unk = np.clip(1 - b.sum(1), 0, 1)
        return b[:, 1] + 0.5 * unk, b

brb = BRB(len(SELECTED))
print("Rule সংখ্যা:", brb.K)

Rule সংখ্যা: 729


In [5]:
def er_analytic(A, beta):
    s = beta.sum(1); one_m = 1 - A * s[None, :]
    f = A[:, :, None] * beta[None] + one_m[:, :, None]
    P = f.prod(1); base = one_m.prod(1); wprod = (1 - A).prod(1)
    mu = 1 / (P.sum(1) - base)
    return mu[:, None] * (P - base[:, None]) / (1 - mu * wprod)[:, None]

def er_recursive(w, beta):
    N = beta.shape[1]
    m = w[0] * beta[0]; mbar = 1 - w[0]; mtil = w[0] * (1 - beta[0].sum())
    for k in range(1, len(w)):
        m2 = w[k] * beta[k]; mbar2 = 1 - w[k]; mtil2 = w[k] * (1 - beta[k].sum())
        mH, mH2 = mbar + mtil, mbar2 + mtil2
        K = 1 / (1 - sum(m[t] * m2[j] for t in range(N) for j in range(N) if j != t))
        m, mtil, mbar = (K * (m * m2 + m * mH2 + mH * m2),
                         K * (mtil * mtil2 + mtil * mbar2 + mbar * mtil2), K * (mbar * mbar2))
    return m / (1 - mbar)

rg = np.random.default_rng(1); worst = 0
for _ in range(300):
    K = rg.integers(2, 40); w = rg.dirichlet(np.ones(K) * rg.uniform(0.3, 2))
    beta = rg.dirichlet(np.ones(2), size=K) * rg.uniform(0.2, 1.0, size=(K, 1))
    worst = max(worst, np.abs(er_analytic(w[None], beta)[0] - er_recursive(w, beta)).max())
print("সর্বোচ্চ পার্থক্য:", worst)

সর্বোচ্চ পার্থক্য: 1.3322676295501878e-15


## 4. Baseline ও BRB (DE ছাড়া)
BRB-র reference point = training data-র ৫/৫০/৯৫ শতাংশক, সব attribute weight সমান।

In [6]:
def mask_random(X, rate, rng):
    Xm = X.copy(); Xm[rng.random(X.shape) < rate] = np.nan
    return Xm

class NBIgnore:
    """Gaussian Naive Bayes (log-scale)। Missing feature-কে likelihood থেকে বাদ দেয়।"""
    def fit(self, X, y):
        Z = slog(X)
        self.mu = np.array([Z[y == c].mean(0) for c in (0, 1)])
        self.var = np.array([Z[y == c].var(0) + 1e-3 for c in (0, 1)])
        self.prior = np.array([(y == c).mean() for c in (0, 1)])
        return self
    def predict_proba(self, X):
        Z = slog(X); ll = np.zeros((len(Z), 2))
        for c in (0, 1):
            t = -0.5 * (np.log(2 * np.pi * self.var[c]) + (Z - self.mu[c]) ** 2 / self.var[c])
            ll[:, c] = np.nansum(t, axis=1) + np.log(self.prior[c])
        ll -= ll.max(1, keepdims=True)
        p = np.exp(ll); return p / p.sum(1, keepdims=True)

def rf(): return make_pipeline(SimpleImputer(strategy="median"),
                               RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1))
def lr(): return make_pipeline(FunctionTransformer(slog), SimpleImputer(strategy="median"),
                               StandardScaler(), LogisticRegression(max_iter=2000))

d = len(SELECTED)
brb = BRB(d)
Q = np.array([[0.05, 0.5, 0.95]] * d)
def make_refs(Z): return np.array([np.quantile(Z[:, i], Q[i]) for i in range(d)])
ONES = np.ones(d)

## 5. Ignorance বিশ্লেষণের নিয়ম
৩০/৫০% missing-এ Gafgyt-এ, BRB-র score (`p`) যেমন আছে তেমনই। শুধু **কোন নমুনা রাখব** তা বদলাই (coverage ৭০% বা ৫০%):

| রাখার নিয়ম | অর্থ |
|---|---|
| `random` | এলোমেলো (তুলনার ভিত্তি) |
| `fewest-missing` | যাদের missing feature কম (সরল, কোনো model লাগে না) |
| `BRB-ignorance` | যাদের BRB-র ignorance কম |
| `BRB-confidence` | যাদের `|p−0.5|` বেশি |

যে নিয়মে রাখা নমুনাগুলোর AUC বেশি, সেটা বেশি কাজের। **`BRB-ignorance` যদি `fewest-missing`-কে না হারায়, ignorance কিছুই যোগ করছে না।**

In [7]:
def selective_auc(y, score, keep_score, coverage, rng):
    k = int(coverage * len(y))
    idx = np.argsort(-(keep_score + rng.random(len(y)) * 1e-9), kind="stable")[:k]
    return roc_auc_score(y[idx], score[idx]) if len(np.unique(y[idx])) == 2 else np.nan

## 6. মূল experiment (প্রতিটা source-এর জন্য)

In [8]:
rows, sel_rows = [], []; t0 = time.time()
for SOURCE in SOURCES:
    XA, yA = make_xy(load_benign(SOURCE), load_attack(SOURCE, "mirai"))
    Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(XA, yA, test_size=0.3, stratify=yA, random_state=SEED)
    Ztr = slog(Xa_tr)
    refs = make_refs(Ztr)
    beta = brb.fit_beliefs(brb.match(Ztr, refs), ya_tr, ONES)
    def brb_out(X):
        Z = slog(X)
        return brb.infer(brb.fill(brb.match(Z, refs), Z), ONES, beta)

    rg_aug = np.random.default_rng(7)
    X_aug = np.vstack([Xa_tr, mask_random(Xa_tr, 0.3, rg_aug), mask_random(Xa_tr, 0.5, rg_aug)])
    y_aug = np.r_[ya_tr, ya_tr, ya_tr]
    base = {"RF-aug": rf().fit(X_aug, y_aug),
            "HGB-d2": HistGradientBoostingClassifier(max_depth=2, max_iter=100, random_state=SEED).fit(Xa_tr, ya_tr),
            "LR": lr().fit(Xa_tr, ya_tr), "NB-ignore": NBIgnore().fit(Xa_tr, ya_tr)}
    print(f"--- SOURCE {SOURCE}: মডেল তৈরি ({time.time()-t0:.0f}s)")

    targets = [("Mirai", t) for t in MIRAI_DEVS if t != SOURCE] + [("Gafgyt", t) for t in range(1, 10) if t != SOURCE]
    for fam, dev in targets:
        X, y = build_scenario(fam, dev)
        for rate in MISSING_RATES:
            acc = {m: [] for m in list(base) + ["BRB"]}
            for r in range(N_REPEATS if rate > 0 else 1):
                Xm = mask_random(X, rate, np.random.default_rng(1000 + r)) if rate > 0 else X
                for m, mdl in base.items():
                    acc[m].append(roc_auc_score(y, mdl.predict_proba(Xm)[:, 1]))
                p, b = brb_out(Xm)
                acc["BRB"].append(roc_auc_score(y, p))
                if fam == "Gafgyt" and rate > 0:
                    rs = np.random.default_rng(5000 + r)
                    u = np.clip(1 - b.sum(1), 0, 1)
                    keeps = {"random": rs.random(len(y)), "fewest-missing": -np.isnan(Xm).sum(1).astype(float),
                             "BRB-ignorance": -u, "BRB-confidence": np.abs(p - 0.5)}
                    for cov in (0.7, 0.5):
                        for kname, ks in keeps.items():
                            sel_rows.append(dict(Source=SOURCE, Device=dev, Missing=int(rate * 100),
                                                 Coverage=cov, Keep=kname,
                                                 AUC=selective_auc(y, p, ks, cov, rs)))
            for m in acc:
                rows.append(dict(Source=SOURCE, Family=fam, Device=dev, Missing=int(rate * 100),
                                 Model=m, AUC=np.mean(acc[m])))
        print(f"SOURCE {SOURCE} | {fam} device {dev} শেষ ({time.time()-t0:.0f}s)")
R = pd.DataFrame(rows); S = pd.DataFrame(sel_rows)

--- SOURCE 2: মডেল তৈরি (31s)
SOURCE 2 | Mirai device 1 শেষ (58s)
SOURCE 2 | Mirai device 4 শেষ (98s)
SOURCE 2 | Mirai device 5 শেষ (122s)
SOURCE 2 | Mirai device 6 শেষ (145s)
SOURCE 2 | Mirai device 8 শেষ (171s)
SOURCE 2 | Mirai device 9 শেষ (196s)
SOURCE 2 | Gafgyt device 1 শেষ (214s)
SOURCE 2 | Gafgyt device 3 শেষ (232s)
SOURCE 2 | Gafgyt device 4 শেষ (251s)
SOURCE 2 | Gafgyt device 5 শেষ (269s)
SOURCE 2 | Gafgyt device 6 শেষ (288s)
SOURCE 2 | Gafgyt device 7 শেষ (306s)
SOURCE 2 | Gafgyt device 8 শেষ (323s)
SOURCE 2 | Gafgyt device 9 শেষ (341s)
--- SOURCE 6: মডেল তৈরি (356s)
SOURCE 6 | Mirai device 1 শেষ (377s)
SOURCE 6 | Mirai device 2 শেষ (400s)
SOURCE 6 | Mirai device 4 শেষ (423s)
SOURCE 6 | Mirai device 5 শেষ (440s)
SOURCE 6 | Mirai device 8 শেষ (460s)
SOURCE 6 | Mirai device 9 শেষ (479s)
SOURCE 6 | Gafgyt device 1 শেষ (495s)
SOURCE 6 | Gafgyt device 2 শেষ (511s)
SOURCE 6 | Gafgyt device 3 শেষ (527s)
SOURCE 6 | Gafgyt device 4 শেষ (545s)
SOURCE 6 | Gafgyt device 5 শেষ (560s)
SOU

## 7. ফলাফল

In [9]:
order = ["RF-aug", "HGB-d2", "LR", "NB-ignore", "BRB"]
print("===== AUC: প্রতিটা source (target device-এর গড়) =====")
print(R.pivot_table(index=["Source", "Family", "Missing"], columns="Model", values="AUC")[order].round(3).to_string())

G = R.groupby(["Source", "Family", "Missing", "Model"])["AUC"].mean().reset_index()
for stat, title in [("mean", "source-গুলোর গড়"), ("min", "source-গুলোর মধ্যে সবচেয়ে খারাপ (worst-case)")]:
    t = G.groupby(["Family", "Missing", "Model"])["AUC"].agg(stat).unstack("Model")[order]
    print(f"\n===== AUC: {title} (এই notebook-এর source {SOURCES}) =====")
    print(t.round(3).to_string())

===== AUC: প্রতিটা source (target device-এর গড়) =====
Model                  RF-aug  HGB-d2     LR  NB-ignore    BRB
Source Family Missing                                         
2      Gafgyt 0         0.723   0.961  0.964      0.817  0.998
              30        0.795   0.796  0.831      0.822  0.957
              50        0.796   0.729  0.802      0.807  0.917
       Mirai  0         1.000   0.999  1.000      0.993  1.000
              30        0.996   0.867  0.995      0.985  0.998
              50        0.985   0.800  0.973      0.971  0.994
6      Gafgyt 0         0.997   0.992  0.908      0.798  0.978
              30        0.928   0.915  0.812      0.807  0.895
              50        0.877   0.877  0.804      0.811  0.891
       Mirai  0         1.000   1.000  1.000      1.000  1.000
              30        0.998   0.999  0.997      0.998  0.999
              50        0.993   0.992  0.982      0.988  0.997
9      Gafgyt 0         0.773   0.976  0.971      0.859  0.859


In [10]:
print("===== Ignorance বিশ্লেষণ: Gafgyt, রাখা নমুনাগুলোর BRB-AUC (বেশি = ভালো নিয়ম) =====")
t = S.pivot_table(index=["Missing", "Coverage"], columns="Keep", values="AUC", aggfunc="mean")
print(t[["random", "fewest-missing", "BRB-ignorance", "BRB-confidence"]].round(3).to_string())
print("\nপ্রতি source-এ (Missing 30%, Coverage 0.7):")
print(S[(S.Missing == 30) & (S.Coverage == 0.7)].pivot_table(index="Source", columns="Keep", values="AUC", aggfunc="mean")
      [["random", "fewest-missing", "BRB-ignorance", "BRB-confidence"]].round(3).to_string())

===== Ignorance বিশ্লেষণ: Gafgyt, রাখা নমুনাগুলোর BRB-AUC (বেশি = ভালো নিয়ম) =====
Keep              random  fewest-missing  BRB-ignorance  BRB-confidence
Missing Coverage                                                       
30      0.5        0.897           0.903          0.862           0.821
        0.7        0.897           0.902          0.855           0.839
50      0.5        0.888           0.904          0.835           0.803
        0.7        0.887           0.901          0.857           0.839

প্রতি source-এ (Missing 30%, Coverage 0.7):
Keep    random  fewest-missing  BRB-ignorance  BRB-confidence
Source                                                       
2        0.957           0.971          0.963           0.949
6        0.895           0.905          0.815           0.808
9        0.837           0.830          0.787           0.760


## 8. Save

In [11]:
os.makedirs("results", exist_ok=True)
R.to_csv("results/part5b_auc.csv", index=False); S.to_csv("results/part5b_selective.csv", index=False)
print("সংরক্ষিত: results/part5b_auc.csv, results/part5b_selective.csv")

সংরক্ষিত: results/part5b_auc.csv, results/part5b_selective.csv


---
## এখন কী?
**File → Download → .ipynb** করে attach করো। এটাই শেষ experiment। এরপর আমি তোমার ৪টা source-এর ফল মিলিয়ে paper-এর claim ঠিক করব, আর লেখা শুরু করব।